In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import gc 

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Finance Transactions"
OUT_DIR = os.path.join(BASE_PATH, "output_powerbi")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Chuyển các ID sang string hoặc int32 thay vì int64 mặc định
dtypes_param = {
    'id': 'int32',
    'card_id': 'int32',
    'client_id': 'int32',
    'mcc': 'str',
    'zip': 'str',
    'merchant_id': 'str'
}

In [ ]:
def clean_money_advanced(series):
    """
    Xử lý tiền tệ :
    - '$12,345.00'  ->  12345.0
    - '($12.00)'    -> -12.0 (Số âm)
    """
    # Chuyển sang string để xử lý chuỗi
    s = series.astype(str)

    # Phát hiện số âm bằng có dấu ngoặc bao quanh không (...)
    # Logic: Nếu chứa '(',  coi đó là số âm
    mask_neg = s.str.contains(r'\(', regex=True)

    # làm sạch: Loại bỏ tất cả ký tự $, ,, (, )
    # Regex [$,()] nghĩa là tìm bất kỳ ký tự nào trong nhóm này và xóa đi
    clean_s = s.str.replace(r'[$,()]', '', regex=True)

    # chuyển sang số (Float)
    # errors='coerce': Nếu gặp lỗi (vd chữ 'Unknown') thì biến thành 0 luôn cho an toàn
    vals = pd.to_numeric(clean_s, errors='coerce').fillna(0)

    # đổi dấu: Nếu lúc nãy mask_neg là True thì nhân với -1
    vals = np.where(mask_neg, -vals, vals)

    return vals

In [ ]:
def load_fraud_labels_correctly(path):
    """
    Xử lý cấu trúc JSON : {"target": {"id": "No", ...}}
    """
    print(f"   Reading JSON from: {path}")
    with open(path, "r") as f:
        data = json.load(f)

    # trích xuất dictionary bên trong key "target"
    if "target" in data:
        raw_dict = data["target"]
        # chuyển đổi: Key -> Transaction_ID, Value -> Target
        # list(raw_dict.items()) tạo ra list các tuple [('10649266', 'No'), ...]
        df = pd.DataFrame(list(raw_dict.items()), columns=["transaction_id", "target"])

        # Chuẩn hóa dữ liệu
        df["transaction_id"] = df["transaction_id"].astype(int) # ID phải là số để merge
        df["target"] = df["target"].apply(lambda x: 1 if str(x).lower() == "yes" else 0)

        print(f"   ✅ Đã load {len(df)} nhãn gian lận.")
        return df
    else:
        raise ValueError("File JSON không có key 'target' như dự kiến!")

In [ ]:
# Load Transactions (lấy cột cần thiết cho nhẹ bớt)
cols_to_load = ['id', 'date', 'client_id', 'card_id', 'amount', 'mcc', 'merchant_city', 'merchant_state', 'use_chip']
trans = pd.read_csv(os.path.join(BASE_PATH, "transactions_data.csv"), usecols=cols_to_load)
trans = trans.rename(columns={"id": "transaction_id"})

# Load Fraud Labels 
fraud = load_fraud_labels_correctly(os.path.join(BASE_PATH, "train_fraud_labels.json"))

# Load Cards & Users
cards = pd.read_csv(os.path.join(BASE_PATH, "cards_data.csv"), usecols=['id', 'card_brand', 'card_type'])
cards = cards.rename(columns={"id": "card_id"})

users = pd.read_csv(os.path.join(BASE_PATH, "users_data.csv"), usecols=['id', 'current_age', 'gender', 'yearly_income'])
users = users.rename(columns={"id": "client_id"})

# Load MCC Map
with open(os.path.join(BASE_PATH, "mcc_codes.json"), "r") as f:
    mcc_map = json.load(f)



In [ ]:
# Merge Transaction + Fraud bằng transaction_id
df = trans.merge(fraud, on="transaction_id", how="left")
df["target"] = df["target"].fillna(0).astype(int) # 0 là sạch, 1 là gian lận

del trans, fraud
gc.collect()

# Merge Cards & Users
df = df.merge(cards, on="card_id", how="left")
df = df.merge(users, on="client_id", how="left")

# Map MCC
df["mcc_desc"] = df["mcc"].astype(str).map(mcc_map).fillna("Unknown")

print(f"   Kích thước sau khi Merge: {df.shape}")

In [ ]:
# Xử lý tiền tệ (Dùng hàm Advanced)
df["amount"] = clean_money_advanced(df["amount"])
if "yearly_income" in df.columns:
    df["yearly_income"] = clean_money_advanced(df["yearly_income"])

# Xử lý thời gian
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour
df["day_name"] = df["date"].dt.day_name()

# Tạo phân khúc (Bins)
df['trans_size'] = pd.cut(df['amount'],
                        bins=[-float('inf'), 0, 50, 200, float('inf')],
                        labels=['Negative/Refund', 'Small', 'Medium', 'Large'])

df['age_group'] = pd.cut(df['current_age'], bins=[0, 25, 45, 65, 100], labels=['<25', '25-45', '45-65', '>65'])


In [ ]:
# User Stats (Dùng cho User Analysis Dashboard)
user_stats = df.groupby('client_id').agg({
    'transaction_id': 'count',
    'amount': 'sum',
    'target': 'sum', # Tổng số lần fraud
    'mcc_desc': lambda x: x.mode()[0] if not x.mode().empty else "Unknown" # Ngành hàng hay mua nhất
}).reset_index()
user_stats.columns = ['client_id', 'total_trans', 'total_spent', 'fraud_count', 'top_category']
user_stats.to_csv(os.path.join(OUT_DIR, "dim_user_risk.csv"), index=False)

#MCC Stats (Dùng cho Fraud Heatmap)
mcc_stats = df.groupby(['mcc', 'mcc_desc']).agg({
    'transaction_id': 'count',
    'amount': 'mean',
    'target': 'mean' # Tỷ lệ Fraud Rate
}).reset_index()
mcc_stats.columns = ['mcc', 'category', 'total_trans', 'avg_amount', 'fraud_rate']
mcc_stats.to_csv(os.path.join(OUT_DIR, "dim_mcc_risk.csv"), index=False)

# Main Fact Table (lấy 500k dòng mới nhất để demo Power BI cho nhẹ)
fact_export = df.sort_values('date', ascending=False).head(500000)
fact_export.to_csv(os.path.join(OUT_DIR, "fact_transactions.csv"), index=False)

print(f"File đã lưu tại: {OUT_DIR}")
print("- fact_transactions.csv: Bảng giao dịch chính")
print("- dim_user_risk.csv: Hồ sơ rủi ro khách hàng")
print("- dim_mcc_risk.csv: Hồ sơ rủi ro ngành hàng")

In [ ]:
# Load lại file gốc (chỉ cột ID)
df_original = pd.read_csv(os.path.join(BASE_PATH, "transactions_data.csv"), usecols=['id'])
print(f"File gốc có {len(df_original)} dòng.")

# Load file Fact vừa xuất ra
df_new = pd.read_csv(os.path.join(OUT_DIR, "fact_transactions.csv"), usecols=['transaction_id'])
print(f"File mới có {len(df_new)} dòng.")

# Lấy 1 ID bất kỳ trong file MỚI
random_id = df_new['transaction_id'].iloc[100] # Lấy dòng thứ 100
print(f"\nTest ID ngẫu nhiên từ file mới: {random_id}")

# Kiểm tra xem ID này có trong file GỐC không?
is_exist = random_id in df_original['id'].values

if is_exist:
    print(f"ID {random_id} CÓ tồn tại trong file gốc (cột 'id').")
    print("=> Dữ liệu hoàn toàn khớp!")
else:
    print(f"Không tìm thấy ID {random_id} trong file gốc.")
    print("=> Cần xem lại quy trình Merge.")

# Kiểm tra ngược: ID file gốc có trong file mới không? (Chỉ check nếu file mới là tập con)
print("\nLưu ý: Không phải ID nào của file gốc cũng có trong file mới (vì đã cắt 500k dòng).")